# House Prices: Advanced Regression Techniques 
## Objective
Predict the final sale price (`SalePrice`) of residential homes in Ames, Iowa,
based on 79 explanatory variables describing almost every aspect of the property
(lot size, quality ratings, garage details, neighborhood, etc.).

## Dataset
- Source: Kaggle "House Prices: Advanced Regression Techniques" competition
- `train.csv`: 1460 rows, 81 columns (80 features + target `SalePrice`)
- `test.csv`: 1459 rows, 80 columns (no target — used for final predictions)
- `data_description.txt`: full definitions of every column and category code

## Imports & Data Loading

Import the libraries needed for data handling, encoding, and modeling,
then load the train and test sets along with the column descriptions.

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Load datasets
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()

Train shape: (1460, 81)
Test shape: (1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## Inspecting Missing Values

Before deciding how to handle missing data, first find out which columns
have missing values and how many. This tells us where to focus — some of
these "missing" values are actually meaningful (e.g. "no garage"), which
we'll confirm against `data_description.txt` in the next step.

In [5]:
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

missing_pct = (missing / len(train)) * 100

missing_summary = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct.round(2)
})

missing_summary

,Missing Count,Missing %
PoolQC,1453,99.52
MiscFeature,1406,96.30
Alley,1369,93.77
Fence,1179,80.75
MasVnrType,872,59.73
FireplaceQu,690,47.26
LotFrontage,259,17.74
GarageType,81,5.55
GarageYrBlt,81,5.55
GarageFinish,81,5.55


## Classifying Missing Values

For each column with missing values, check `data_description.txt` to see
whether `NA` is a documented category (meaning "this feature doesn't exist
for this house") or whether it's genuinely unknown data. This split
determines how we handle each column.

In [7]:
with open('data_description.txt', 'r') as f:
    description_text = f.read()

none_means_no_feature = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'MasVnrType'
]

genuinely_missing = ['LotFrontage', 'GarageYrBlt', 'MasVnrArea', 'Electrical']

all_missing_cols = set(missing_summary.index)
accounted_for = set(none_means_no_feature + genuinely_missing)

print("Missing columns not yet classified:", all_missing_cols - accounted_for)
print("Classified columns not actually missing:", accounted_for - all_missing_cols)

Missing columns not yet classified: set()
Classified columns not actually missing: set()


## Filling Missing Values

Apply the two strategies decided in the previous step:
- For columns where `NA` means "no feature," fill with the string `"None"`
  so it becomes a real, explicit category instead of a gap.
- For genuinely missing columns, impute using a method appropriate to
  that specific column (grouped median, mode, or a logical default).

We apply the same fills to both `train` and `test` so they stay consistent.

In [8]:
for df in [train, test]:
    for col in none_means_no_feature:
        df[col] = df[col].fillna("None")

    # LotFrontage: fill with median LotFrontage of the same Neighborhood
    df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'] \
                           .transform(lambda x: x.fillna(x.median()))

    df['MasVnrArea'] = df['MasVnrArea'].fillna(0)

    df['GarageYrBlt'] = df['GarageYrBlt'].fillna(0)

    df['Electrical'] = df['Electrical'].fillna(train['Electrical'].mode()[0])

check_cols = none_means_no_feature + genuinely_missing
print("Remaining missing values in targeted columns:")
print(train[check_cols].isnull().sum().sum())
print(test[check_cols].isnull().sum().sum())

Remaining missing values in targeted columns:
0
0


## Fixing Incorrect Data Types

**MSSubClass** is stored as a number (e.g. 20, 60, 190), but it's actually a
categorical building-class code, the values don't represent a quantity,
so a higher number doesn't mean "more" of anything. If left as an integer,
any model would wrongly treat it as continuous/ordinal. Cast it to a
string so it's treated as a category during encoding.

In [9]:
print("Before:", train['MSSubClass'].dtype)
print("Unique values:", sorted(train['MSSubClass'].unique()))

for df in [train, test]:
    df['MSSubClass'] = df['MSSubClass'].astype(str)

print("After:", train['MSSubClass'].dtype)

Before: int64
Unique values: [20, 30, 40, 45, 50, 60, 70, 75, 80, 85, 90, 120, 160, 180, 190]
After: object


## Classifying Categorical Columns: Ordinal vs. Nominal

Before encoding, every categorical column needs to be sorted into one of
two groups:
- **Ordinal**: the categories have a genuine rank (e.g. quality ratings
  from Poor to Excellent). These will use ordinal encoding with an
  explicit order we define ourselves.
- **Nominal**: the categories have no inherent order (e.g. neighborhood
  names, roof style). These will use one-hot encoding.


In [11]:
ordinal_cols = {
    'ExterQual':    ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'ExterCond':    ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtQual':     ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtCond':     ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'HeatingQC':    ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'KitchenQual':  ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'FireplaceQu':  ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageQual':   ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageCond':   ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'PoolQC':       ['None', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtExposure': ['None', 'No', 'Mn', 'Av', 'Gd'],
    'BsmtFinType1': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'BsmtFinType2': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'GarageFinish': ['None', 'Unf', 'RFn', 'Fin'],
    'Functional':   ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
    'LotShape':     ['IR3', 'IR2', 'IR1', 'Reg'],
    'LandSlope':    ['Sev', 'Mod', 'Gtl'],
    'PavedDrive':   ['N', 'P', 'Y'],
    'Fence':        ['None', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv'],
    'Utilities':    ['NoSeWa', 'AllPub'],
}

categorical_cols = train.select_dtypes(include='object').columns.tolist()
nominal_cols = [c for c in categorical_cols if c not in ordinal_cols]

print(f"Ordinal columns ({len(ordinal_cols)}):", list(ordinal_cols.keys()))
print(f"\nNominal columns ({len(nominal_cols)}):", nominal_cols)

Ordinal columns (20): ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'GarageFinish', 'Functional', 'LotShape', 'LandSlope', 'PavedDrive', 'Fence', 'Utilities']

Nominal columns (24): ['MSSubClass', 'MSZoning', 'Street', 'Alley', 'LandContour', 'LotConfig', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating', 'CentralAir', 'Electrical', 'GarageType', 'MiscFeature', 'SaleType', 'SaleCondition']


## Applying Ordinal Encoding

For each ordinal column, map its categories to integers using the order
we defined above (not **LabelEncoder**, which would assign order
alphabetically and ignore the real ranking).

In [18]:
from sklearn.preprocessing import OrdinalEncoder

for col, order in ordinal_cols.items():
    encoder = OrdinalEncoder(categories=[order], handle_unknown='use_encoded_value', unknown_value=-1)

    train[col] = encoder.fit_transform(train[[col]])
    test[col] = encoder.transform(test[[col]])

# Spot check a couple of columns
train[list(ordinal_cols.keys())].head()

ValueError: could not convert string to float: 'Po'